# Week 3 - Superstore Sales Analysis

Analysing the Superstore dataset using SQL. The goal is to use subqueries, CTEs and
window functions to answer a set of business questions about customers and their sales.

I loaded the CSV into a SQLite database, split it into three tables (`customers`,
`products`, `orders`) and then ran the queries from the notebook itself.

## Step 1 - Setup

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

In [2]:
csv_path = Path("../data/Sample - Superstore.csv")
df = pd.read_csv(csv_path, encoding="latin-1")
df.shape

(9994, 21)

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
    .str.lower()
)
df.columns.tolist()

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit']

In [4]:
df["order_date"] = pd.to_datetime(df["order_date"], format="%m/%d/%Y").dt.strftime("%Y-%m-%d")
df["ship_date"] = pd.to_datetime(df["ship_date"], format="%m/%d/%Y").dt.strftime("%Y-%m-%d")
df[["order_date", "ship_date"]].head()

,order_date,ship_date
0,2016-11-08,2016-11-11
1,2016-11-08,2016-11-11
2,2016-06-12,2016-06-16
3,2015-10-11,2015-10-18
4,2015-10-11,2015-10-18


In [5]:
conn = sqlite3.connect("../output/superstore.db")
df.to_sql("superstore_raw", conn, if_exists="replace", index=False)
pd.read_sql("SELECT COUNT(*) AS rows FROM superstore_raw", conn)

,rows
0,9994


Now I create the three tables from `superstore_raw` using `SELECT DISTINCT`.
The schema lives in `sql/schema.sql` so I just read and run it here.

In [6]:
schema = Path("../sql/schema.sql").read_text()
conn.executescript(schema)
conn.commit()

In [7]:
for t in ["customers", "products", "orders"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", conn)["n"][0]
    print(t, n)

customers 793
products 1894
orders 9994


A small helper so every query below is just one line.

In [8]:
def run(query):
    return pd.read_sql(query, conn)

## Step 2 - Required Queries

**1. Orders where sales are greater than the average sales** *(subquery)*

In [9]:
run("""
SELECT order_id, customer_id, sales
FROM orders
WHERE sales > (SELECT AVG(sales) FROM orders)
ORDER BY sales DESC
""").head(10)

,order_id,customer_id,sales
0,CA-2014-145317,SM-20320,22638.480
1,CA-2016-118689,TC-20980,17499.950
2,CA-2017-140151,RB-19360,13999.960
3,CA-2017-127180,TA-21385,11199.968
4,CA-2017-166709,HL-15040,10499.970
5,CA-2016-117121,AB-10105,9892.740
6,CA-2014-116904,SC-20095,9449.950
7,US-2016-107440,BS-11365,9099.930
8,CA-2016-158841,SE-20110,8749.950
9,CA-2016-143714,CC-12370,8399.976


**2. Highest sales order for each customer** *(subquery)*

In [10]:
run("""
SELECT o.customer_id, o.order_id, o.sales
FROM orders o
WHERE o.sales = (
    SELECT MAX(o2.sales)
    FROM orders o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY o.sales DESC
""").head(10)

,customer_id,order_id,sales
0,SM-20320,CA-2014-145317,22638.480
1,TC-20980,CA-2016-118689,17499.950
2,RB-19360,CA-2017-140151,13999.960
3,TA-21385,CA-2017-127180,11199.968
4,HL-15040,CA-2017-166709,10499.970
5,AB-10105,CA-2016-117121,9892.740
6,SC-20095,CA-2014-116904,9449.950
7,BS-11365,US-2016-107440,9099.930
8,SE-20110,CA-2016-158841,8749.950
9,CC-12370,CA-2016-143714,8399.976


**3. Total sales for each customer** *(CTE)*

In [11]:
run("""
WITH customer_sales AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name, ROUND(cs.total_sales, 2) AS total_sales
FROM customer_sales cs
JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY cs.total_sales DESC
""").head(10)

,customer_name,total_sales
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57
5,Ken Lonsdale,14175.23
6,Sanjit Chand,14142.33
7,Hunter Lopez,12873.30
8,Sanjit Engle,12209.44
9,Christopher Conant,12129.07


**4. Customers whose total sales are above average** *(CTE + subquery)*

In [12]:
run("""
WITH customer_sales AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name, ROUND(cs.total_sales, 2) AS total_sales
FROM customer_sales cs
JOIN customers c ON c.customer_id = cs.customer_id
WHERE cs.total_sales > (SELECT AVG(total_sales) FROM customer_sales)
ORDER BY cs.total_sales DESC
""").head(10)

,customer_name,total_sales
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57
5,Ken Lonsdale,14175.23
6,Sanjit Chand,14142.33
7,Hunter Lopez,12873.30
8,Sanjit Engle,12209.44
9,Christopher Conant,12129.07


**5. Rank all customers based on total sales** *(window function)*

In [13]:
run("""
WITH customer_sales AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name,
       ROUND(cs.total_sales, 2) AS total_sales,
       RANK() OVER (ORDER BY cs.total_sales DESC) AS sales_rank
FROM customer_sales cs
JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY sales_rank
""").head(10)

,customer_name,total_sales,sales_rank
0,Sean Miller,25043.05,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.34,3
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,5
5,Ken Lonsdale,14175.23,6
6,Sanjit Chand,14142.33,7
7,Hunter Lopez,12873.30,8
8,Sanjit Engle,12209.44,9
9,Christopher Conant,12129.07,10


**6. Row number for each order within a customer** *(window function + PARTITION BY)*

In [14]:
run("""
SELECT customer_id, order_id, order_date, sales,
       ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_seq
FROM orders
ORDER BY customer_id, order_seq
""").head(15)

,customer_id,order_id,order_date,sales,order_seq
0,AA-10315,CA-2014-128055,2014-03-31,673.568,1
1,AA-10315,CA-2014-128055,2014-03-31,52.980,2
2,AA-10315,CA-2014-138100,2014-09-15,14.940,3
3,AA-10315,CA-2014-138100,2014-09-15,14.560,4
4,AA-10315,CA-2015-121391,2015-10-04,26.960,5
5,AA-10315,CA-2016-103982,2016-03-03,3930.072,6
6,AA-10315,CA-2016-103982,2016-03-03,2.304,7
7,AA-10315,CA-2016-103982,2016-03-03,431.976,8
8,AA-10315,CA-2016-103982,2016-03-03,41.720,9
9,AA-10315,CA-2017-147039,2017-06-29,362.940,10


**7. Top 3 customers based on total sales** *(window function)*

In [15]:
run("""
WITH customer_sales AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
),
ranked AS (
    SELECT c.customer_name, cs.total_sales,
           RANK() OVER (ORDER BY cs.total_sales DESC) AS sales_rank
    FROM customer_sales cs
    JOIN customers c ON c.customer_id = cs.customer_id
)
SELECT customer_name, ROUND(total_sales, 2) AS total_sales, sales_rank
FROM ranked
WHERE sales_rank <= 3
""")

,customer_name,total_sales,sales_rank
0,Sean Miller,25043.05,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.34,3


## Step 3 - Final Combined Query

Customer name, total sales and rank together, using a JOIN, a CTE and a window function.

In [16]:
run("""
WITH customer_sales AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name,
       ROUND(cs.total_sales, 2) AS total_sales,
       RANK() OVER (ORDER BY cs.total_sales DESC) AS sales_rank
FROM customer_sales cs
JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY sales_rank
""").head(10)

,customer_name,total_sales,sales_rank
0,Sean Miller,25043.05,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.34,3
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,5
5,Ken Lonsdale,14175.23,6
6,Sanjit Chand,14142.33,7
7,Hunter Lopez,12873.30,8
8,Sanjit Engle,12209.44,9
9,Christopher Conant,12129.07,10


## Mini Project - Customer Sales Insights

**1. Top 5 customers**

In [17]:
run("""
WITH customer_sales AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name, ROUND(cs.total_sales, 2) AS total_sales
FROM customer_sales cs
JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY cs.total_sales DESC
LIMIT 5
""")

,customer_name,total_sales
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57


**2. Bottom 5 customers**

In [18]:
run("""
WITH customer_sales AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name, ROUND(cs.total_sales, 2) AS total_sales
FROM customer_sales cs
JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY cs.total_sales ASC
LIMIT 5
""")

,customer_name,total_sales
0,Thais Sissman,4.83
1,Lela Donovan,5.30
2,Carl Jackson,16.52
3,Mitch Gastineau,16.74
4,Roy Skaria,22.33


**3. Customers who made only one order**

In [19]:
one_order = run("""
SELECT c.customer_name, COUNT(DISTINCT o.order_id) AS order_count
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
HAVING COUNT(DISTINCT o.order_id) = 1
ORDER BY c.customer_name
""")
print("customers with a single order:", len(one_order))
one_order.head(10)

customers with a single order: 12


,customer_name,order_count
0,Anemone Ratner,1
1,Anthony O'Donnell,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1
5,Lela Donovan,1
6,Mitch Gastineau,1
7,Patricia Hirasaki,1
8,Ricardo Emerson,1
9,Roland Murray,1


**4. Customers with above-average sales**

In [20]:
above_avg = run("""
WITH customer_sales AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name, ROUND(cs.total_sales, 2) AS total_sales
FROM customer_sales cs
JOIN customers c ON c.customer_id = cs.customer_id
WHERE cs.total_sales > (SELECT AVG(total_sales) FROM customer_sales)
ORDER BY cs.total_sales DESC
""")
print("above average customers:", len(above_avg))
above_avg.head(10)

above average customers: 294


,customer_name,total_sales
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57
5,Ken Lonsdale,14175.23
6,Sanjit Chand,14142.33
7,Hunter Lopez,12873.30
8,Sanjit Engle,12209.44
9,Christopher Conant,12129.07


**5. Highest order value per customer**

In [21]:
run("""
WITH order_totals AS (
    SELECT customer_id, order_id, SUM(sales) AS order_value
    FROM orders
    GROUP BY customer_id, order_id
)
SELECT c.customer_name, ROUND(MAX(ot.order_value), 2) AS highest_order_value
FROM order_totals ot
JOIN customers c ON c.customer_id = ot.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY highest_order_value DESC
""").head(10)

,customer_name,highest_order_value
0,Sean Miller,23661.23
1,Tamara Chand,18336.74
2,Raymond Buch,14052.48
3,Tom Ashbrook,13716.46
4,Becky Martin,10539.90
5,Hunter Lopez,10499.97
6,Sanjit Chand,9900.19
7,Adrian Barton,9892.74
8,Bill Shonely,9135.19
9,Sanjit Engle,8805.04


## Notes

A few things I noticed while working through the data:

- Sales are very skewed. A small group of customers sits well above the average, while
  most customers are below it, so the mean total sales is higher than the median.
- Sean Miller tops the list at around 25k, well ahead of the next customer, mostly on
  the back of a few very large orders rather than a steady stream of small ones.
- Only 12 of the 793 customers placed a single order, so repeat buying is actually the
  norm here rather than the exception.

`order_date` and `ship_date` were stored as `MM/DD/YYYY` strings in the CSV, so I
converted them to `YYYY-MM-DD` before loading. That keeps the `ORDER BY order_date`
in the window function chronological instead of sorting them as plain text.

In [22]:
conn.close()